# Aerynza AI — Free GPU Launcher (Colab / Kaggle)

Run each cell in order. Set the runtime to a **GPU** before starting.

This trains a Qwen 0.6B LoRA on the full 640-capability proven master in one click.

In [ ]:
import torch
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — stop and enable GPU runtime')

## 1. Clone Ascension AI

In [ ]:
!git clone https://github.com/acensionlifeos-dev/ascension-ai.git
%cd ascension-ai

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements-training.txt

## 3. Build the proven master curriculum
This merges all non-category reviewed conversations into one 640-capability training set.

In [ ]:
!python -u scripts/build_proven_master.py

## 4. Patch for T4 / P100 free GPUs
Qwen uses bfloat16 by default, but free Colab/Kaggle cards usually need float16.

In [ ]:
from pathlib import Path
path = Path('scripts/train_qwen_ascension_lora.py')
text = path.read_text()
text = text.replace('torch.bfloat16 if torch.cuda.is_available() else torch.float32', 'torch.float16 if torch.cuda.is_available() else torch.float32')
text = text.replace('bf16=torch.cuda.is_available(),', 'bf16=False,
        fp16=torch.cuda.is_available(),')
path.write_text(text)
print('Patched to float16 for free GPUs')

## 5. Train Qwen 0.6B LoRA on 640-capability master
Change `--epochs` to 4.0 if you want more training.

In [ ]:
!python -u scripts/train_qwen_ascension_lora.py \
    --model Qwen/Qwen3-0.6B \
    --curriculum "evals/training/ascension_product_v159_proven_master.jsonl" \
    --output-dir checkpoints/qwen3_0_6b_grow001_adapter \
    --epochs 2.0 \
    --learning-rate 1.0e-4 \
    --max-length 512

## 6. Package the new adapter for download

In [ ]:
!zip -r aerynza_grow001_adapter.zip checkpoints/qwen3_0_6b_grow001_adapter
try:
    from google.colab import files
    files.download('aerynza_grow001_adapter.zip')
except Exception:
    print('Kaggle users: download aerynza_grow001_adapter.zip from the Output tab. Colab users: the file is in the file browser on the left.')